# ByT5 Final Submission

Single byt5-base model fine-tuned on all 17 languages (no per-language modules)
plus a deterministic copy-fallback post-processing layer.

Best dev-phase codabench score: **52.01** weighted ERR (macro-average 54.28; trained on train+val combined).

**Run order:**
1. Cell 1 -- training utility (`train_byt5(...)`)
2. Cell 2 -- final-phase training on train+val combined 
3. Cell 3 -- load final checkpoint, define `byt5_predict_words` helper
4. Cell 4 -- build train+val copy-fallback count table, define `apply_fallback`
5. Cell 5 -- predict on dev-pub test, apply fallback, save submission zip


In [ ]:
# === ByT5 training utilities (replaces train_byt5.py) ===
# Defines the train_byt5(...) entry point used by all the training cells below.

import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from torch.utils.data import WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)


def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_word_pairs(split):
    langs, raws, norms = [], [], []
    for item in split:
        lang = item["lang"]
        for r, n in zip(item["raw"], item["norm"]):
            langs.append(lang)
            raws.append(r)
            norms.append(n)
    return Dataset.from_dict({"lang": langs, "raw": raws, "norm": norms})


def _make_tokenize_fn(tokenizer, max_input_len, max_target_len):
    def fn(batch):
        inputs = [f"{l}: {r}" for l, r in zip(batch["lang"], batch["raw"])]
        targets = batch["norm"]
        model_inputs = tokenizer(inputs, max_length=max_input_len, truncation=True, padding=False)
        labels = tokenizer(text_target=targets, max_length=max_target_len, truncation=True, padding=False)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return fn


def _detect_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _build_per_lang_subset(pairs, n_per_lang, seed):
    rng = np.random.default_rng(seed)
    by_lang = {}
    for i, lang in enumerate(pairs["lang"]):
        by_lang.setdefault(lang, []).append(i)
    chosen = []
    for lang, idxs in by_lang.items():
        if len(idxs) > n_per_lang:
            picked = rng.choice(idxs, size=n_per_lang, replace=False)
            chosen.extend(int(x) for x in picked)
        else:
            chosen.extend(idxs)
    chosen.sort()
    return pairs.select(chosen)


def _word_err(raws, golds, preds):
    n = len(raws)
    if n == 0:
        return 0.0, 0.0, 0.0
    correct_sys = sum(1 for p, g in zip(preds, golds) if p == g)
    correct_lai = sum(1 for r, g in zip(raws, golds) if r == g)
    acc_sys = correct_sys / n
    acc_lai = correct_lai / n
    denom = 1.0 - acc_lai
    err = (acc_sys - acc_lai) / denom if denom > 0 else 0.0
    return acc_lai, acc_sys, err


class _WeightedSamplerTrainer(Seq2SeqTrainer):
    def __init__(self, *args, train_sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_sample_weights = train_sample_weights

    def _get_train_sampler(self, *args, **kwargs):
        if self.train_sample_weights is None:
            return super()._get_train_sampler(*args, **kwargs)
        g = torch.Generator()
        g.manual_seed(self.args.seed)
        return WeightedRandomSampler(
            weights=self.train_sample_weights,
            num_samples=len(self.train_dataset),
            replacement=True,
            generator=g,
        )


def train_byt5(
    model_name="google/byt5-base",
    dataset_name="weerayut/multilexnorm2026-dev-pub",
    output_dir="./checkpoints/byt5-base-final",
    epochs=3,
    batch_size=64,
    eval_batch_size=128,
    grad_accum=1,
    lr=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_input_len=64,
    max_target_len=64,
    seed=42,
    fp16=False,
    bf16=None,
    num_workers=None,
    early_stop_patience=999,        # effectively disabled (fixed-epoch training)
    save_total_limit=2,
    resume=False,
    oversample_weights=None,
    target_langs=None,
    eval_per_lang_n=200,
    gradient_checkpointing=False,
    combine_train_val=False,        # final-phase: merge train+val, no val signal
):
    """Fine-tune ByT5 on MultiLexNorm 2026. Outputs best model to output_dir/final/.

    Set combine_train_val=True for the final-phase recipe (UFAL 2021): merge
    train+val into training data, disable evaluation/early-stop, save the last
    checkpoint.
    """
    device = _detect_device()
    if bf16 is None:
        bf16 = device == "cuda"
    if num_workers is None:
        num_workers = 4 if device == "cuda" else 0
    print(f"[device] {device} (fp16={fp16}, bf16={bf16}, num_workers={num_workers})")

    _set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"[load] dataset = {dataset_name}")
    ds = load_dataset(dataset_name)
    print(f"[load] splits = {list(ds.keys())}")
    print(f"[load] train = {len(ds['train'])}  val = {len(ds['validation'])}")

    if target_langs:
        targets = set(target_langs)
        ds = ds.filter(lambda x: x["lang"] in targets)
        print(f"[filter] keep langs={sorted(targets)} -> train={len(ds['train'])} val={len(ds['validation'])}")

    if combine_train_val:
        print("[combine] using train + validation as training data (final-phase recipe)")
        train_source = concatenate_datasets([ds["train"], ds["validation"]])
    else:
        train_source = ds["train"]

    print("[prep] flattening sentence -> word pairs ...")
    train_pairs = _build_word_pairs(train_source)
    val_pairs = _build_word_pairs(ds["validation"]) if not combine_train_val else None
    if val_pairs is not None:
        print(f"[prep] train pairs = {len(train_pairs)}  val pairs = {len(val_pairs)}")
    else:
        print(f"[prep] train pairs = {len(train_pairs)}  (no val: combined into train)")

    train_counts = Counter(train_pairs["lang"])
    print(f"[stats] train pairs by lang: {dict(sorted(train_counts.items()))}")

    sample_weights = None
    if oversample_weights:
        sample_weights = [oversample_weights.get(l, 1.0) for l in train_pairs["lang"]]
        per_lang_w = Counter()
        for l, w in zip(train_pairs["lang"], sample_weights):
            per_lang_w[l] += w
        total_w = sum(sample_weights)
        ratios = {l: round(w / total_w * 100, 2) for l, w in sorted(per_lang_w.items())}
        print(f"[oversample] weights override: {oversample_weights}")
        print(f"[oversample] expected per-lang sampling fraction (%): {ratios}")

    print(f"[load] tokenizer + model = {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name, use_safetensors=True)

    tokenize_fn = _make_tokenize_fn(tokenizer, max_input_len, max_target_len)
    train_tok = train_pairs.map(
        tokenize_fn, batched=True, remove_columns=train_pairs.column_names, desc="tokenize train",
    )

    eval_tok = None
    eval_langs_arr = eval_raws_arr = eval_norms_arr = None
    if val_pairs is not None:
        print(f"[eval] building per-lang val subset (n_per_lang={eval_per_lang_n})")
        eval_subset = _build_per_lang_subset(val_pairs, eval_per_lang_n, seed)
        eval_counts = Counter(eval_subset["lang"])
        print(f"[eval] subset size = {len(eval_subset)}  per-lang = {dict(sorted(eval_counts.items()))}")
        eval_langs_arr = list(eval_subset["lang"])
        eval_raws_arr = list(eval_subset["raw"])
        eval_norms_arr = list(eval_subset["norm"])
        eval_tok = eval_subset.map(
            tokenize_fn, batched=True, remove_columns=eval_subset.column_names, desc="tokenize eval_subset",
        )

    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    def compute_metrics(eval_preds):
        preds = eval_preds.predictions
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.asarray(preds)
        if (preds == -100).any():
            preds = np.where(preds == -100, tokenizer.pad_token_id, preds)
        decoded = tokenizer.batch_decode(preds, skip_special_tokens=True)
        m = min(len(decoded), len(eval_langs_arr))
        by_lang = {}
        all_r, all_n, all_p = [], [], []
        for i in range(m):
            lang = eval_langs_arr[i]
            slot = by_lang.setdefault(lang, ([], [], []))
            slot[0].append(eval_raws_arr[i])
            slot[1].append(eval_norms_arr[i])
            slot[2].append(decoded[i])
            all_r.append(eval_raws_arr[i])
            all_n.append(eval_norms_arr[i])
            all_p.append(decoded[i])
        out = {}
        for lang in sorted(by_lang):
            r, n, p = by_lang[lang]
            _, _, err = _word_err(r, n, p)
            out[f"err_{lang}"] = round(err * 100, 4)
        _, _, err_avg = _word_err(all_r, all_n, all_p)
        out["err_average"] = round(err_avg * 100, 4)
        return out

    # Build training args; in combine_train_val mode there is no val so we
    # turn off evaluation / early stopping / best-checkpoint selection.
    args_kw = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        logging_steps=100,
        save_total_limit=save_total_limit,
        predict_with_generate=True,
        generation_max_length=max_target_len,
        generation_num_beams=1,
        fp16=fp16,
        bf16=bf16 and not fp16,
        dataloader_num_workers=num_workers,
        report_to=["none"],
        seed=seed,
        gradient_checkpointing=gradient_checkpointing,
        gradient_checkpointing_kwargs={"use_reentrant": False} if gradient_checkpointing else None,
    )
    if combine_train_val:
        args_kw["eval_strategy"] = "no"
        args_kw["save_strategy"] = "no"           # only the final save_model() below
        args_kw["load_best_model_at_end"] = False
    else:
        args_kw["eval_strategy"] = "epoch"
        args_kw["save_strategy"] = "epoch"
        args_kw["load_best_model_at_end"] = True
        args_kw["metric_for_best_model"] = "err_average"
        args_kw["greater_is_better"] = True

    training_args = Seq2SeqTrainingArguments(**args_kw)

    trainer_kw = dict(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        processing_class=tokenizer,
        data_collator=collator,
        train_sample_weights=sample_weights,
    )
    if eval_tok is not None:
        trainer_kw["eval_dataset"] = eval_tok
        trainer_kw["compute_metrics"] = compute_metrics
        trainer_kw["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=early_stop_patience)]
    trainer = _WeightedSamplerTrainer(**trainer_kw)

    resume_flag = False
    if resume:
        ckpts = sorted(output_dir.glob("checkpoint-*"))
        if ckpts:
            print(f"[resume] found {len(ckpts)} checkpoint(s); latest = {ckpts[-1].name}")
            resume_flag = True
        else:
            print("[resume] no checkpoint found, starting from scratch")
    print(f"[train] starting ... (resume={resume_flag})")
    trainer.train(resume_from_checkpoint=resume_flag)

    final_dir = output_dir / "final"
    print(f"[save] best model -> {final_dir}")
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    # Free disk: best checkpoint is now in final/, drop intermediate checkpoint-* dirs.
    import shutil
    for d in output_dir.iterdir():
        if d.is_dir() and d.name.startswith("checkpoint-"):
            shutil.rmtree(d)
            print(f"[cleanup] removed {d}")
    print("[done]")


In [ ]:
# Train byt5-base UNIFIED on train + validation combined (final-phase recipe).
# epochs=3 chosen because dev-phase ablation showed 5 epochs over-trains on ko
# (codabench dev: 5ep train+val gave ko=-3.85 vs baseline best-of-5 ko=11.54).
# Skip if checkpoint already exists.

import os

FINAL_OUT = "./checkpoints/byt5-base-final-3ep"

if os.path.exists(f"{FINAL_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {FINAL_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=FINAL_OUT,
        epochs=3,                       # was 5 -- 3 hits dev-best sweet spot
        batch_size=64,
        num_workers=0,
        combine_train_val=True,        # train+val merged for max data (UFAL recipe)
    )


In [ ]:
# Load the final checkpoint and define byt5_predict_words for inference.
# Auto-detects CUDA / MPS (Apple Silicon) / CPU.

from pathlib import Path
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

FINAL_CKPT = "./checkpoints/byt5-base-final-3ep/final"
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

byt5_tok = AutoTokenizer.from_pretrained(FINAL_CKPT)
byt5_model = T5ForConditionalGeneration.from_pretrained(FINAL_CKPT).to(device).eval()
print(f"loaded {FINAL_CKPT} on {device}")


def byt5_predict_words(words, langs, batch_size=256, max_input=64, max_target=64, num_beams=1):
    """Batched word-level prediction."""
    preds = []
    for i in range(0, len(words), batch_size):
        cw, cl = words[i:i+batch_size], langs[i:i+batch_size]
        enc = byt5_tok([f"{l}: {w}" for l, w in zip(cl, cw)],
                       return_tensors="pt", padding=True,
                       truncation=True, max_length=max_input).to(device)
        with torch.no_grad():
            out = byt5_model.generate(**enc, max_new_tokens=max_target, num_beams=num_beams)
        preds.extend(byt5_tok.batch_decode(out, skip_special_tokens=True))
    return preds


In [ ]:
# Copy fallback post-processing.
# Builds a per-language raw -> {norm: count} table from train + val and defines
# apply_fallback(). Per-lang thresholds were tuned on val using TRAIN-ONLY counts
# (no leakage). At test time we use train+val counts because test is disjoint
# from both splits.

from collections import defaultdict
from datasets import load_dataset

byt5_data = load_dataset("weerayut/multilexnorm2026-dev-pub")


def build_counts(items):
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    for item in items:
        lang = item["lang"]
        for r, n in zip(item["raw"], item["norm"]):
            counts[lang][r][n] += 1
    return {lang: {raw: dict(d) for raw, d in c.items()} for lang, c in counts.items()}


fallback_counts = build_counts(list(byt5_data["train"]) + list(byt5_data["validation"]))

# (min_count, threshold) per high-LAI language (tuned on val, train-only counts)
FALLBACK_CFG = {
    "ko": (1, 0.60),
    "ja": (1, 0.40),
    "th": (1, 0.65),
    "sr": (1, 0.30),
    "en": (1, 0.55),
    "hr": (1, 0.50),
    "sl": (1, 0.50),
    "vi": (1, 0.40),
}


def apply_fallback(raw, pred, lang, counts=fallback_counts, cfg=FALLBACK_CFG):
    if pred == raw:
        return pred                       # model didn't change -> nothing to veto
    if lang not in cfg:
        return pred                       # no fallback for this lang
    min_count, threshold = cfg[lang]
    lang_c = counts.get(lang, {})
    if raw not in lang_c:
        return pred
    total = sum(lang_c[raw].values())
    if total < min_count:
        return pred
    keep_count = lang_c[raw].get(raw, 0)
    return raw if keep_count / total >= threshold else pred


print({lang: len(fallback_counts[lang]) for lang in sorted(fallback_counts)})


In [ ]:
# Predict on dev-pub test, apply copy fallback, save final submission zip.

import zipfile
from pathlib import Path

DEV_SAVE = "outputs/submission_dev_byt5_final_fb"
DEV_ZIP = "outputs/submission_dev_byt5_final_fb.zip"

# dev-pub test split (byt5_data was loaded from weerayut/multilexnorm2026-dev-pub
# in the previous cell, so we reuse it instead of reloading)
test_df = byt5_data["test"].to_pandas()
flat_words, flat_langs, sent_lens = [], [], []
for _, row in test_df.iterrows():
    flat_words.extend(row["raw"])
    flat_langs.extend([row["lang"]] * len(row["raw"]))
    sent_lens.append(len(row["raw"]))
print(f"dev-pub test: {len(flat_words)} words across {len(test_df)} sentences")

# Model inference
preds = byt5_predict_words(flat_words, flat_langs)

# Apply copy fallback per word
preds_fb = [apply_fallback(r, p, l) for r, p, l in zip(flat_words, preds, flat_langs)]

# Regroup back to per-sentence
test_preds, cursor = [], 0
for n in sent_lens:
    test_preds.append(preds_fb[cursor:cursor+n])
    cursor += n
test_df["pred"] = test_preds

Path(DEV_SAVE).mkdir(parents=True, exist_ok=True)
test_df.to_json(f"{DEV_SAVE}/predictions.json", orient="records")
with zipfile.ZipFile(DEV_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(f"{DEV_SAVE}/predictions.json", arcname="predictions.json")
print(f"saved final submission: {DEV_ZIP}")
test_df[["raw", "pred", "lang"]].head(3)
